## Explore Metadata
In this file I show how to explore the RKD metadata (data/rkd_metadata.xml)

In [9]:
import xml.etree.ElementTree as ET

# The data is structured in an XML-tree
tree = ET.parse('../data/rkd_metadata.xml')
root = tree.getroot()

In [13]:
######## List for every painting the face images #############
from pathlib import Path

afbeeldingen_path = Path("../data/paintings")
gezichten_path = Path("../data/faces/")

afbeeldingen_files = list(afbeeldingen_path.glob("*.jp*g"))
gezichten_files = list(gezichten_path.glob("*.jp*g"))

afbeeldingen = {file.stem: file.name for file in afbeeldingen_files}
afbeeldingen_gezichten = {file.stem: [] for file in afbeeldingen_files}

for gezicht in gezichten_files:
    stem = gezicht.stem
    afbeelding = stem.split('-')[0]
    afbeeldingen_gezichten[afbeelding].append(stem)

print('List for every painting the face images:', len(afbeeldingen_gezichten), afbeeldingen_gezichten)

afbeeldingen_een_gezicht = {k: v for k, v in afbeeldingen_gezichten.items() if len(v) == 1}
afbeeldingen_meerdere_gezichten = {k: v for k, v in afbeeldingen_gezichten.items() if len(v) > 1}

print('List for every painting with only one face detected:', len(afbeeldingen_een_gezicht), afbeeldingen_een_gezicht)
print('List for every painting with more than one face detected:', len(afbeeldingen_meerdere_gezichten), afbeeldingen_meerdere_gezichten)

List for every painting the face images: 1673 {'0000005601': ['0000005601-0'], '0000005801': ['0000005801-0', '0000005801-1', '0000005801-2', '0000005801-3'], '0000010961': ['0000010961-0'], '0000011054': ['0000011054-0'], '0000011120': ['0000011120-0'], '0000012545': ['0000012545-0'], '0000021972': ['0000021972-0'], '0000026500': ['0000026500-0'], '0000026801': ['0000026801-0'], '0000028287': ['0000028287-0'], '0000029100': ['0000029100-0'], '0000030449': ['0000030449-0'], '0000030651': ['0000030651-0'], '0000030851': ['0000030851-0'], '0000036062': ['0000036062-0'], '0000036101': ['0000036101-0'], '0000037417': ['0000037417-0'], '0000037451': ['0000037451-0'], '0000037851': ['0000037851-0'], '0000047151': ['0000047151-0'], '0000052089': ['0000052089-0', '0000052089-1', '0000052089-10', '0000052089-11', '0000052089-12', '0000052089-13', '0000052089-14', '0000052089-15', '0000052089-16', '0000052089-17', '0000052089-18', '0000052089-19', '0000052089-2', '0000052089-20', '0000052089-21'

In [14]:
########### Lijst met beschikbare plaatjes ###########
prirefs = {}
for record in root:
    priref = record.findtext('priref')
    prirefs[priref] = []
    for media in record.findall('.//original_file_name'):
        file_name = media.text
        if file_name in afbeeldingen:
            prirefs[priref].append(file_name)

prirefs_beschikbaar = {k: v for k, v in prirefs.items() if v != []}
print(len(prirefs_beschikbaar), prirefs_beschikbaar)

afbeelding_priref = {v_item: k for k, v_list in prirefs_beschikbaar.items() for v_item in v_list}
print(len(afbeelding_priref), afbeelding_priref)

1673 {'743': ['1001171753'], '1850': ['1001337107'], '1864': ['1000313301'], '1994': ['0000323151'], '2001': ['0000259663'], '2035': ['1001131501'], '2953': ['0001526287'], '3046': ['0001526301'], '3371': ['1001197151'], '3638': ['0000030651'], '4082': ['1001335151'], '4163': ['0000005801'], '4996': ['1001335328'], '5385': ['1001196069'], '5412': ['0000372110'], '5430': ['0000010961'], '5617': ['IB00020251'], '5630': ['1000278851'], '5636': ['1000312419'], '5637': ['IB00066351'], '5655': ['1001085001'], '5693': ['0000011054'], '5735': ['1001247251'], '5748': ['0000054101'], '5790': ['0000351764'], '5825': ['0000357855'], '5855': ['0000011120'], '5900': ['IB00046809'], '6428': ['1001213050'], '6634': ['1000312901'], '6869': ['IB00065401'], '7101': ['1001252014'], '7115': ['0000323051'], '7143': ['0000352685'], '8117': ['1001636321'], '8285': ['1001695801'], '8333': ['1001203724'], '10592': ['IB00044031'], '10594': ['0000129342'], '10706': ['0000372101'], '10715': ['1000313886'], '10719'

In [16]:
from pprint import pprint

# Variable to go down the list one by one
nr = None
img = '0000123929'
gez = None

### Print information for each painting
for i, (afbeelding, gezichten) in enumerate(afbeeldingen_een_gezicht.items()):
    if i == nr or afbeelding == img or gezichten == img:
        print(afbeelding, gezichten)
        for record in root:
            priref = record.findtext('priref')
            if priref == afbeelding_priref[afbeelding]:
                print(f"priref: {priref}")
                print(f"afbeelding: {afbeelding}")
                print(f'gezicht afbeeldingen: {gezichten}')

                for media in record.findall('.//benaming_kunstwerk'):
                    print('benaming_kunstwerk:', media.text)

                for media in record.findall('.//titel_engels'):
                    print('titel_engels:', media.text)

                # for media in record.findall('.//toeschrijving/'):
                #     print('titel_engels:', media.text)

                for toeschrijving in record.findall('toeschrijving'):
                    naam = toeschrijving.find('naam/kunstenaarsnaam')
                    status = toeschrijving.find('status')
                    if naam != '\n':
                        print(f'Kunstenaar: {naam.text} (status: {status.text})')

                print('\n')
                ids = len([p for p in record.findall('.//persoonsnummer') if p.text != '\n'])

                personen = [{
                        'person_id': '',
                        'full_name': '',
                        'description': '',
                        'status_identification_portret': '',
                        'certainty_identification_portret': '',
                        'birthdate': '',
                        'deathdate': '',
                    } for _ in range(ids)]
                c = 0
                for media in record.findall('.//persoonsnummer'):
                    if media.text != '\n':
                        persoonsnummer = media.text
                        personen[c]['person_id'] = persoonsnummer
                        c += 1
                c = 0
                for media in record.findall('.//naam_volledig'):
                    if media.text != '\n':
                        persoonsnummer = media.text
                        personen[c]['full_name'] = persoonsnummer
                        c += 1
                c=0
                for media in record.findall('.//persoonsbeschrijving'):
                    if media.text != '\n':
                        persoonsnummer = media.text
                        personen[c]['description'] = persoonsnummer
                        c += 1

                c = 0
                for media in record.findall('.//status_identificatie_portret'):
                    if media.text != '\n':
                        persoonsnummer = media.text
                        personen[c]['status_identification_portret'] = persoonsnummer
                        c += 1
                c = 0
                for media in record.findall('.//zekerheid_identificatie_portret'):
                    if media.text != '\n':
                        persoonsnummer = media.text
                        personen[c]['certainty_identification_portret'] = persoonsnummer
                        c += 1
                c = 0
                for media in record.findall('.//persoonsnummer_geboortedatum_begin'):
                    if media.text != '\n':
                        persoonsnummer = media.text
                        personen[c]['birthdate'] = persoonsnummer
                        c += 1
                c = 0
                for media in record.findall('.//persoonsnummer_sterfdatum_begin'):
                    if media.text != '\n':
                        persoonsnummer = media.text
                        personen[c]['deathdate'] = persoonsnummer
                        c += 1
                pprint(personen)

0000123929 ['0000123929-0']
priref: 104349
afbeelding: 0000123929
gezicht afbeeldingen: ['0000123929-0']
benaming_kunstwerk: Portret van een onbekende vrouw
benaming_kunstwerk: Portret van een onbekende man
titel_engels: Portrait of an unknown woman
Kunstenaar: Bailly, David (status: huidig)
Kunstenaar: Vries, Abraham de (status: verworpen)


[{'birthdate': '1613-10-06',
  'certainty_identification_portret': None,
  'deathdate': None,
  'description': None,
  'full_name': 'Bosch, Johanna Margaretha de',
  'person_id': '116805',
  'status_identification_portret': 'verworpen'}]


In [ ]:
# Print everything
for r, record in enumerate(root.iter('record')):
    for a, alpha in enumerate(record):
        print(r, a, alpha.tag, alpha.text)
        for b, beta in enumerate(alpha):
            print('\t', r, a, b, beta.tag, beta.text)
            for g, gamma in enumerate(beta):
                print('\t\t', r, a, b, g, gamma.tag, gamma.text)